# 16 — Inter-annotator agreement

Assesses coding reliability by re-annotating a pooled sample of the RQ1 corpus (simple random + rare-positive enrichment, n = 350, seed 42) and comparing the two independent codings with raw agreement, Cohen's κ, and exact three-label pattern agreement. The first cell reconstructs the blind sample and its hidden key from `labelled_comments_final.csv`; the analysis cells compute agreement from the completed re-annotation.


In [ ]:
# COMPLETE CONSTRUCTION OF THE FINAL POOLED n=350 SAMPLE

from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Paths and source corpus
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "processed"

SOURCE_PATH = DATA_DIR / "labelled_comments_final.csv"
COMPLETED_PATH = DATA_DIR / "inter_annotator_agreement_blind_350.csv"

corpus_raw = pd.read_csv(SOURCE_PATH, encoding="utf-8-sig")

# Reproduce the final deduplicated human-coded corpus
corpus = (
    corpus_raw
    .drop_duplicates(subset="text_clean", keep="first")
    .reset_index(drop=True)
)

label_columns = ["label_1A", "label_1B", "label_2"]

for column in label_columns:
    corpus[column] = pd.to_numeric(
        corpus[column],
        errors="raise"
    ).astype(int)

assert len(corpus_raw) == 2031
assert len(corpus) == 2026
assert corpus[label_columns].isin([0, 1]).all().all()


# ---------------------------------------------------------
# 2. Reusable deterministic sampling function
# ---------------------------------------------------------

def choose_rows(frame, n, rng):
    """Randomly sample after sorting IDs for reproducibility."""

    ordered = (
        frame
        .sort_values("comment_id", kind="stable")
        .reset_index(drop=True)
    )

    if len(ordered) < n:
        raise ValueError(
            f"Only {len(ordered)} eligible rows available; {n} required."
        )

    positions = rng.choice(
        len(ordered),
        size=n,
        replace=False
    )

    return ordered.iloc[positions].copy()


# ---------------------------------------------------------
# 3. Initial simple random sample: n=250
# ---------------------------------------------------------

random_250 = (
    corpus
    .sample(n=250, random_state=42)
    .sample(frac=1, random_state=123)
    .reset_index(drop=True)
)

random_250.insert(
    0,
    "agreement_id",
    [f"IAA{i:04d}" for i in range(1, 251)]
)

random_250["selection_group"] = "simple_random"

random_250_ids = set(
    random_250["comment_id"].astype(str)
)


# ---------------------------------------------------------
# 4. First enriched selection stage: 36 rare positives
# ---------------------------------------------------------

eligible_stage1 = corpus.loc[
    ~corpus["comment_id"].astype(str).isin(random_250_ids)
].copy()

stage1_pools = {
    "positive_1B_and_2": eligible_stage1.loc[
        (eligible_stage1["label_1B"] == 1)
        & (eligible_stage1["label_2"] == 1)
    ],

    "positive_1B_only": eligible_stage1.loc[
        (eligible_stage1["label_1B"] == 1)
        & (eligible_stage1["label_2"] == 0)
    ],

    "positive_2_without_1A": eligible_stage1.loc[
        (eligible_stage1["label_1A"] == 0)
        & (eligible_stage1["label_1B"] == 0)
        & (eligible_stage1["label_2"] == 1)
    ],

    "positive_2_with_1A": eligible_stage1.loc[
        (eligible_stage1["label_1A"] == 1)
        & (eligible_stage1["label_1B"] == 0)
        & (eligible_stage1["label_2"] == 1)
    ],
}

stage1_targets = {
    "positive_1B_and_2": 6,
    "positive_1B_only": 8,
    "positive_2_without_1A": 15,
    "positive_2_with_1A": 7,
}

stage1_order = [
    "positive_1B_and_2",
    "positive_1B_only",
    "positive_2_without_1A",
    "positive_2_with_1A",
]

rng_stage1 = np.random.default_rng(20260902)
stage1_parts = []

for group in stage1_order:
    selected = choose_rows(
        stage1_pools[group],
        stage1_targets[group],
        rng_stage1
    )

    selected["selection_group"] = group
    stage1_parts.append(selected)

enriched_36 = pd.concat(
    stage1_parts,
    ignore_index=True
)

enriched_36 = (
    enriched_36
    .iloc[rng_stage1.permutation(len(enriched_36))]
    .reset_index(drop=True)
)

enriched_36.insert(
    0,
    "agreement_id",
    [f"IAA_P{i:04d}" for i in range(1, 37)]
)


# ---------------------------------------------------------
# 5. Add 64 rows to produce the final enriched n=100
# ---------------------------------------------------------

already_selected_ids = (
    random_250_ids
    | set(enriched_36["comment_id"].astype(str))
)

eligible_stage2 = corpus.loc[
    ~corpus["comment_id"].astype(str).isin(already_selected_ids)
].copy()

stage2_pools = {
    "positive_1B_only": eligible_stage2.loc[
        (eligible_stage2["label_1A"] == 1)
        & (eligible_stage2["label_1B"] == 1)
        & (eligible_stage2["label_2"] == 0)
    ],

    "positive_2_without_1A": eligible_stage2.loc[
        (eligible_stage2["label_1A"] == 0)
        & (eligible_stage2["label_1B"] == 0)
        & (eligible_stage2["label_2"] == 1)
    ],

    "positive_2_with_1A": eligible_stage2.loc[
        (eligible_stage2["label_1A"] == 1)
        & (eligible_stage2["label_1B"] == 0)
        & (eligible_stage2["label_2"] == 1)
    ],

    "positive_1A_only_control": eligible_stage2.loc[
        (eligible_stage2["label_1A"] == 1)
        & (eligible_stage2["label_1B"] == 0)
        & (eligible_stage2["label_2"] == 0)
    ],

    "all_negative_control": eligible_stage2.loc[
        (eligible_stage2["label_1A"] == 0)
        & (eligible_stage2["label_1B"] == 0)
        & (eligible_stage2["label_2"] == 0)
    ],
}

stage2_targets = {
    "positive_1B_only": 16,
    "positive_2_without_1A": 1,
    "positive_2_with_1A": 1,
    "positive_1A_only_control": 23,
    "all_negative_control": 23,
}

stage2_order = [
    "positive_1B_only",
    "positive_2_without_1A",
    "positive_2_with_1A",
    "positive_1A_only_control",
    "all_negative_control",
]

rng_stage2 = np.random.default_rng(20260903)
stage2_parts = []

for group in stage2_order:
    selected = choose_rows(
        stage2_pools[group],
        stage2_targets[group],
        rng_stage2
    )

    selected["selection_group"] = group
    stage2_parts.append(selected)

additional_64 = pd.concat(
    stage2_parts,
    ignore_index=True
)

additional_64.insert(
    0,
    "agreement_id",
    [f"IAA_P{i:04d}" for i in range(37, 101)]
)

# Combine and shuffle the complete enriched supplement
enriched_100 = pd.concat(
    [enriched_36, additional_64],
    ignore_index=True
)

enriched_100 = (
    enriched_100
    .iloc[rng_stage2.permutation(len(enriched_100))]
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Create one final pooled n=350 sample
# ---------------------------------------------------------

sample_350 = pd.concat(
    [random_250, enriched_100],
    ignore_index=True
)

assert len(random_250) == 250
assert len(enriched_100) == 100
assert len(sample_350) == 350
assert sample_350["comment_id"].nunique() == 350
assert sample_350["text_clean"].nunique() == 350

assert set(random_250["comment_id"]).isdisjoint(
    set(enriched_100["comment_id"])
)


# ---------------------------------------------------------
# 7. Demonstrate that original labels were removed
# ---------------------------------------------------------

blind_template_350 = sample_350[
    ["agreement_id", "comment_id", "text_clean"]
].copy()

blind_template_350["label_1A_annotator2"] = pd.NA
blind_template_350["label_1B_annotator2"] = pd.NA
blind_template_350["label_2_annotator2"] = pd.NA
blind_template_350["annotator2_notes"] = pd.NA


# ---------------------------------------------------------
# 8. Verify against the actual completed n=350 file
# ---------------------------------------------------------

completed_350 = pd.read_csv(
    COMPLETED_PATH,
    encoding="utf-8-sig"
)

constructed_ids = set(
    sample_350["comment_id"].astype(str)
)

completed_ids = set(
    completed_350["comment_id"].astype(str)
)

assert constructed_ids == completed_ids, (
    "The reconstructed sample does not match the completed sample."
)


# ---------------------------------------------------------
# 9. Reproducibility audit
# ---------------------------------------------------------

enriched_counts = {
    label: int(enriched_100[label].sum())
    for label in label_columns
}

pooled_counts = {
    label: int(sample_350[label].sum())
    for label in label_columns
}

rare_positive_rows = int(
    (
        (enriched_100["label_1B"] == 1)
        | (enriched_100["label_2"] == 1)
    ).sum()
)

control_rows = len(enriched_100) - rare_positive_rows

print("Raw labelled corpus:", len(corpus_raw))
print("Deduplicated corpus:", len(corpus))
print("Final pooled sample:", len(sample_350))
print("Overlap between components: 0")

print("\nEnriched supplement label counts:")
print(enriched_counts)

print("\nRare-positive enriched rows:", rare_positive_rows)
print("Control enriched rows:", control_rows)

print("\nOriginal positive counts in pooled n=350:")
print(pooled_counts)

print("\nMatches completed coded file: True")

print("\nTime-1 positive counts in final pooled n=350:")
print(pooled_counts)

## Manual re-annotation handoff

The construction cell above writes the blind 350-comment sample and its hidden key. The 350 comments are re-coded **by hand**, blind to the original labels; the completed file is then compared against the key below. As with the RQ1 corpus and the RQ2 audit, the comment-level data files are part of the assumed source data and are not published in the public repository (see the README).


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Handles VS Code starting the notebook inside /notebooks
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"

repeat = pd.read_csv(
    DATA_DIR / "inter_annotator_agreement_blind_350.csv",
    encoding="utf-8-sig"
)

original = pd.read_csv(
    DATA_DIR / "labelled_comments_final.csv",
    encoding="utf-8-sig"
)

print("Repeated coding:", repeat.shape)
print("Original dataset:", original.shape)

print("\nRepeated-coding columns:")
print(repeat.columns.tolist())

print("\nMissing repeated labels:")
print(
    repeat[
        [
            "label_1A_annotator2",
            "label_1B_annotator2",
            "label_2_annotator2",
        ]
    ].isna().sum()
)

In [ ]:
assert len(repeat) == 350, "The repeat sample does not contain 350 rows."
assert repeat["agreement_id"].nunique() == 350
assert repeat["comment_id"].nunique() == 350

print("All 350 repeated annotations loaded successfully.")

In [ ]:
import numpy as np
import unicodedata

# The key identifies which rows belonged to the original 250
key = pd.read_csv(
    DATA_DIR / "inter_annotator_agreement_key_DO_NOT_OPEN.csv",
    encoding="utf-8-sig"
)

# Rename the repeated judgements as Time 2
repeat_time2 = repeat.rename(columns={
    "text_clean": "text_clean_time2",
    "label_1A_annotator2": "label_1A_time2",
    "label_1B_annotator2": "label_1B_time2",
    "label_2_annotator2": "label_2_time2",
})

# Select and rename the original judgements as Time 1
original_time1 = original[
    ["comment_id", "text_clean", "label_1A", "label_1B", "label_2"]
].rename(columns={
    "text_clean": "text_clean_time1",
    "label_1A": "label_1A_time1",
    "label_1B": "label_1B_time1",
    "label_2": "label_2_time1",
})

assert original_time1["comment_id"].is_unique

# Match each repeated judgement to its original judgement
iaa = repeat_time2.merge(
    original_time1,
    on="comment_id",
    how="left",
    validate="one_to_one"
)

# Identify the original and enriched portions without relying on row order
original_250_ids = set(key["comment_id"])

iaa["iaa_sample"] = np.where(
    iaa["comment_id"].isin(original_250_ids),
    "original_250",
    "enriched_100"
)

# Check whether exactly the same text was presented both times
def normalise_text(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    return " ".join(value.split())

time1_text = iaa["text_clean_time1"].map(normalise_text)
time2_text = iaa["text_clean_time2"].map(normalise_text)

iaa["text_integrity"] = [
    "exact" if original_text == repeated_text
    else "time2_truncated" if original_text.startswith(repeated_text)
    else "different"
    for original_text, repeated_text in zip(time1_text, time2_text)
]

iaa["comparable_flag"] = (iaa["text_integrity"] == "exact").astype(int)

# Final checks
time1_labels = [
    "label_1A_time1",
    "label_1B_time1",
    "label_2_time1",
]

assert len(iaa) == 350
assert iaa[time1_labels].notna().all().all()
assert iaa["comment_id"].nunique() == 350

print("Merged rows:", len(iaa))
print("\nSample composition:")
print(iaa["iaa_sample"].value_counts())

print("\nText integrity:")
print(iaa["text_integrity"].value_counts())

In [ ]:
# Inspect 2 different rows

from difflib import SequenceMatcher

different_rows = iaa[
    iaa["text_integrity"] == "different"
].copy()

different_rows["time1_length"] = (
    different_rows["text_clean_time1"].astype(str).str.len()
)

different_rows["time2_length"] = (
    different_rows["text_clean_time2"].astype(str).str.len()
)

different_rows["similarity"] = [
    SequenceMatcher(None, str(time1), str(time2)).ratio()
    for time1, time2 in zip(
        different_rows["text_clean_time1"],
        different_rows["text_clean_time2"]
    )
]

for _, row in different_rows.iterrows():
    print("=" * 80)
    print("Agreement ID:", row["agreement_id"])
    print("Comment ID:", row["comment_id"])
    print("Sample:", row["iaa_sample"])
    print("Similarity:", round(row["similarity"], 3))
    print("\nTIME 1:")
    print(row["text_clean_time1"])
    print("\nTIME 2:")
    print(row["text_clean_time2"])
    print()

In [ ]:
# Correct classification for this
minor_format_ids = ["IAA0024", "IAA0200"]

iaa.loc[
    iaa["agreement_id"].isin(minor_format_ids),
    "text_integrity"
] = "minor_format_difference"

iaa["comparable_flag"] = iaa["text_integrity"].isin(
    ["exact", "minor_format_difference"]
).astype(int)

print(iaa["text_integrity"].value_counts())
print("\nComparable rows:", iaa["comparable_flag"].sum())

# Keep only the directly comparable rows, then remove the temporary fields
iaa = iaa[iaa["comparable_flag"] == 1].copy()
iaa = iaa.drop(
    columns=["iaa_sample", "comparable_flag"],
    errors="ignore"
)

assert len(iaa) == 341

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# Remove the temporary sample-splitting fields
iaa = iaa.drop(
    columns=["iaa_sample", "comparable_flag"],
    errors="ignore"
)

assert len(iaa) == 341

LABEL_PAIRS = {
    "1A": ("label_1A_time1", "label_1A_time2"),
    "1B": ("label_1B_time1", "label_1B_time2"),
    "2":  ("label_2_time1", "label_2_time2"),
}

# Ensure labels are numeric
for time1_col, time2_col in LABEL_PAIRS.values():
    iaa[time1_col] = pd.to_numeric(
        iaa[time1_col], errors="raise"
    ).astype(int)

    iaa[time2_col] = pd.to_numeric(
        iaa[time2_col], errors="raise"
    ).astype(int)


results = []

for label, (time1_col, time2_col) in LABEL_PAIRS.items():

    time1 = iaa[time1_col]
    time2 = iaa[time2_col]

    tn, fp, fn, tp = confusion_matrix(
        time1,
        time2,
        labels=[0, 1]
    ).ravel()

    positive_denominator = (2 * tp) + fp + fn
    negative_denominator = (2 * tn) + fp + fn

    results.append({
        "label": label,
        "n": len(iaa),
        "time1_positive_n": int(time1.sum()),
        "time2_positive_n": int(time2.sum()),
        "time1_positive_rate": time1.mean(),
        "time2_positive_rate": time2.mean(),
        "raw_agreement": (time1 == time2).mean(),
        "cohens_kappa": cohen_kappa_score(time1, time2),
        "positive_agreement": (
            2 * tp / positive_denominator
            if positive_denominator else np.nan
        ),
        "negative_agreement": (
            2 * tn / negative_denominator
            if negative_denominator else np.nan
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    })

metrics = pd.DataFrame(results)


# Agreement across all three labels simultaneously
time1_columns = [
    "label_1A_time1",
    "label_1B_time1",
    "label_2_time1",
]

time2_columns = [
    "label_1A_time2",
    "label_1B_time2",
    "label_2_time2",
]

iaa["exact_pattern_agreement"] = (
    iaa[time1_columns].to_numpy()
    == iaa[time2_columns].to_numpy()
).all(axis=1)

exact_pattern_agreement = iaa["exact_pattern_agreement"].mean()

display(metrics.round(3))


print(
    f"\nExact three-label agreement: "
    f"{exact_pattern_agreement:.1%} (n={len(iaa)})"
)
